## $^{14}$N( $^{17}$F, $^{18}$Ne)$^{13}$C transfer reaction

We will use the [transfer input file from the fresco website](https://www.fresco.org.uk/examples/B5-example-tr.in) as a template.

In this reaction, a proton from $^{14}$N is transfered to $^{17}$F, to give $^{18}$Ne, leaving a residual $^{13}$C nucleus. From the [Fresco website](https://www.fresco.org.uk/guide/fresco-starting.pdf):

> Transfer reactions are often used to extract structure information to input in astrophysical simulations. Here we
consider the 14N(17F,18Ne)13C transfer reaction at 10 MeV per nucleon. This reaction was measured with the
aim of extracting the asymptotic normalization coefficient of specific states in 18Ne which in turn provides a
significant part of the rate for 17F(p,γ). The proton capture reaction on 17F appears in the rp-process in novae
environments. The ratio of the proton capture rate and the decay rate of 17F is also very important for the
understanding of galactic 17O, 18O and 15N. 


The interactions involved are:
- the entrance channel interaction between $^{14}$N and  $^{17}$F,
- the exit channel interaction between $^{18}$Ne and $^{13}$C,
- the binding potential for the single $p$ in $^{17}$F,
- the binding potential for the single $p$ in $^{13}$C,
- the core-core potential between $^{17}$F and $^{13}$C,

We will compare to experimental data from [EXFOR Entry C1137](https://www-nds.iaea.org/exfor/servlet/X4sGetSubent?reqx=2507&subID=121137003&plus=1), which was measured at Oak Ridge National Laboratory and first [reported in the literature in 2004](https://www.sciencedirect.com/science/article/pii/S0375947404010188?via%3Dihub)

In [1]:
import os
import shutil
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import inspect
import pandas as pd
import json

with open("MatplotlibEsthetics.json", "r") as fptr:
    esthetics = json.load(fptr)

plt.style.use(esthetics["style"])
FONTSIZE      = esthetics["fontsize"]
TICK_FONTSIZE = esthetics["tick_fontsize"]
MARKERSIZE    = esthetics["markersize"]
LINEWIDTH     = esthetics["linewidth"]

In [2]:
import bfrescox

In [3]:
EXAMPLE_DIR = Path("./Transfer_template/")
TEMPLATE_FILE_PATH = Path("./Transfer_template/B5-example-tr.template") 
INPUT_FILE_PATH = Path("./Transfer_template/B5-example-tr.in") 

## Let's run the input file exactly as is from the website

In [4]:
with open(INPUT_FILE_PATH, "r") as temp:
    input_file = temp.read()

print("Frescox input file from https://www.fresco.org.uk/examples/B5-example-tr.out:")
print("-----------------------------------")
print(input_file)

Frescox input file from https://www.fresco.org.uk/examples/B5-example-tr.out:
-----------------------------------
n14(f17,ne18)c13 @ 170 MeV; 
NAMELIST
 &FRESCO hcm=0.03 rmatch=40 rintp=0.20 hnl=0.1 rnl=5.00 centre=0.0 
	 jtmin=0.0  jtmax=120 absend=-1.0
	 thmin=0.00 thmax=40.00 thinc=0.10
	 iter=1 nnu=36
	 chans=1 xstabl=1 
	 elab=170.0  /

 &PARTITION namep='f17'  massp=17. zp=9 namet='n14'  masst=14. zt=7 nex=1  /
 &STATES jp=2.5 bandp=1 ep=0.0 cpot=1 jt=1.0 bandt=1 et=0.0000  /

 &PARTITION namep='ne18' massp=18. zp=10 namet='c13'  masst=13. zt=6 qval=3.6286 nex=2  /
 &STATES jp=0. bandp=1 ep=0.0  cpot=2 jt=0.5 bandt=1 et=0.0000  /
 &STATES jp=4. bandp=1 ep=3.376  cpot=2 copyt  /


 &partition /

 &POT kp=1 ap=17.000 at=14.000 rc=1.3  /
 &POT kp=1 type=1 p1=37.2 p2=1.2 p3=0.6  p4=21.6 p5=1.2 p6=0.69  /

 &POT kp=2 ap=18.000 at=13.000 rc=1.3  /
 &POT kp=2 type=1 p1=37.2 p2=1.2 p3=0.6  p4=21.6 p5=1.2 p6=0.69  /

 &POT kp=3 at=17 rc=1.2  /
 &POT kp=3 type=1 p1=50.00 p2=1.2 p3=0.65   /

In [10]:
# Create the frescox input file by filling in the user-defined template with parameters
cfg = bfrescox.Configuration.from_template(
                    INPUT_FILE_PATH,
                    EXAMPLE_DIR.joinpath("frescox.in"),
                    {},
                    overwrite=True,
                )

In [11]:
%%time
bfrescox.run_simulation(cfg, EXAMPLE_DIR.joinpath("frescox.out"), cwd=EXAMPLE_DIR, overwrite=True)

CPU times: user 4.07 ms, sys: 901 μs, total: 4.97 ms
Wall time: 4.02 ms


FileNotFoundError: [Errno 2] No such file or directory: '/home/beyerk/Projects/Bfrescox/book/notebooks/Transfer_template/frescox.in'

In [ ]:
results = bfrescox.parse_fort16(EXAMPLE_DIR.joinpath("fort.16"))
results.keys()

In [ ]:
plt.plot(
    results["channel_3"]["Theta_deg"], 
    results["channel_3"]["sigma_mb_sr"],
    label="Frescox"
)

plt.xlim([0,20])
plt.xlabel(r"$\theta$ [deg]")
plt.ylabel(r"$d\sigma/d\Omega$ [mb/Sr]")

## Now let's look at the template and see how to change the parameters

In [ ]:
with open(TEMPLATE_FILE_PATH, "r") as temp:
    template_file = temp.read()

print("Frescox Transfer Template:")
print("-----------------------------------")
print(template_file)

In [ ]:
base_params = {
    "rC_entrance" : 1.3,
    "V_entrance": 37.2,
    "r_entrance": 1.2,
    "a_entrance" : 0.6,
    "W_entrance": 21.6,
    "rw_entrance": 1.2,
    "aw_entrance" : 0.69,
    "rC_exit" : 1.3,
    "V_exit": 37.2,
    "r_exit": 1.2,
    "a_exit" : 0.6,
    "W_exit": 21.6,
    "rw_exit": 1.2,
    "aw_exit" : 0.69,
    "rC_p17F": 1.2,
    "V_p17F": 50, 
    "r_p17F": 1.2,
    "a_p17F":  0.65,
    "Vso_p17F": 6, 
    "rso_p17F": 1.2,
    "aso_p17F":  0.65,
    "rC_p13C": 1.2,
    "V_p13C": 50, 
    "r_p13C": 1.2,
    "a_p13C":  0.65,
    "Vso_p13C": 6, 
    "rso_p13C": 1.2,
    "aso_p13C":  0.65,
    "rC_17F_13C": 1.3,
    "V_17F_13C": 37.2,
    "r_17F_13C": 1.2, 
    "a_17F_13C": 0.6, 
    "W_17F_13C": 21.6, 
    "rw_17F_13C": 1.2, 
    "aw_17F_13C": 0.69,  
    "BE_p_17F" : 3.922,
    "BE_p_13C": 7.5506,
}

In [ ]:
# Create the frescox input file by filling in the user-defined template with parameters
cfg = bfrescox.Configuration.from_template(
                    TEMPLATE_FILE_PATH,
                    EXAMPLE_DIR.joinpath("frescox.in"),
                    base_params,
                    overwrite=True,
                )

In [ ]:
bfrescox.run_simulation(cfg, EXAMPLE_DIR.joinpath("frescox.out"), cwd=EXAMPLE_DIR, overwrite=True)

In [ ]:
results = bfrescox.parse_fort16(EXAMPLE_DIR.joinpath("fort.16"))
display(results["channel_1"].head())

In [ ]:
plt.plot(
    results["channel_2"]["Theta_deg"], 
    results["channel_2"]["sigma_mb_sr"],
    label="Fresco"
)
plt.legend()
plt.xlim([0,20])
plt.xlabel(r"$\theta$ [deg]")
plt.ylabel(r"$d\sigma/d\Omega$ [mb/Sr]")